# Proyecto Integrador - Minería de Datos I

**Notebook 01 - Inspección Inicial**

**Objetivo**

En este notebook se realiza una primera exploración del conjunto de datos con el fin de conocer su estructura, identificar los tipos de variables y detectar posibles problemas de calidad, como valores faltantes, registros duplicados o inconsistencias. En esta etapa no se realizan modificaciones sobre los datos; únicamente se recopila evidencia que servirá de base para las etapas posteriores del proyecto.

In [1]:
# Importación de librerías

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Configuración para visualizar todas las columnas
pd.set_option("display.max_columns", None)

# Mejor formato para números decimales
pd.options.display.float_format = "{:.2f}".format

In [2]:
# Carga del dataset

ruta = "/content/drive/MyDrive/PI_Mineria_Datos_1/data/raw/streaming_users_dirty.json"

df = pd.read_json(ruta)

print("Dataset cargado correctamente.")

Dataset cargado correctamente.


## Descripción del dataset

El conjunto de datos contiene información de usuarios de una plataforma de streaming. Entre las variables disponibles se incluyen datos demográficos, tipo de suscripción, tiempo mensual de visualización, país de residencia, género favorito, fecha del último acceso y cantidad de tickets de soporte.


In [3]:
# Vista previa del dataset
df.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.80,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.40,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.00,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.40,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.80,Perú,Thriller,2020-09-30,1


### Observación

Se muestran los primeros registros del dataset para conocer la estructura de las variables y verificar que la carga de los datos se realizó correctamente.

In [4]:
# Últimos registros
df.tail()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
8155,10923,23,Premium,1161.40,Colombia,Romance,2023-05-15,0
8156,16525,27,Básico,436.20,Uruguay,Documental,2018-09-06,4
8157,11222,13,Estándar,1321.80,México,Documental,2019-02-08,0
8158,15613,38,Estándar,835.70,Brasil,Drama,2022-02-05,0
8159,16912,25,Estándar,1468.70,Argentina,Romance,2022-08-12,3


### Observación

La visualización de los últimos registros permite verificar que la estructura del dataset se mantiene de forma consistente hasta el final del archivo.

In [5]:
# Dimensiones del dataset

print("Dimensiones del dataset (filas, columnas):")
display(df.shape)

filas, columnas = df.shape

print(f"\nCantidad de registros: {filas}")
print(f"Cantidad de variables: {columnas}")

Dimensiones del dataset (filas, columnas):


(8160, 8)


Cantidad de registros: 8160
Cantidad de variables: 8


Conocer las dimensiones del dataset permite tener una primera idea del volumen de información disponible para el análisis.

In [6]:
# Información general

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   object 
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   object 
 5   favorite_genre            7920 non-null   object 
 6   last_login_date           7840 non-null   object 
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


La información general permite identificar el tipo de dato de cada variable y detectar, de manera preliminar, la existencia de valores faltantes.

In [7]:
# Variables del dataset

variables = pd.DataFrame({
    "Variable": df.columns,
    "Tipo de dato": df.dtypes.values
})

variables

,Variable,Tipo de dato
0,user_id,int64
1,age,int64
2,subscription_plan,object
3,monthly_watch_time_mins,float64
4,country,object
5,favorite_genre,object
6,last_login_date,object
7,customer_support_tickets,int64


La tabla anterior resume las variables presentes en el conjunto de datos junto con su tipo de dato. Esta información permite identificar qué variables son numéricas y cuáles son categóricas, facilitando la planificación de las etapas posteriores del análisis.

In [8]:
# Variables numéricas

df.describe().T

,count,mean,std,min,25%,50%,75%,max
user_id,8160.00,13995.43,2310.81,10000.00,11987.75,13998.50,15997.25,17999.00
age,8160.00,34.10,14.51,-5.00,25.00,33.00,42.00,150.00
monthly_watch_time_mins,7967.00,1107.35,5310.44,-120.00,489.20,757.40,1045.70,99999.00
customer_support_tickets,8160.00,1.80,11.33,-1.00,0.00,1.00,1.00,150.00


Las estadísticas descriptivas permiten conocer la distribución general de las variables numéricas e identificar posibles valores extremos que serán analizados en etapas posteriores.

In [9]:
# Valores faltantes

nulos = pd.DataFrame({
    "Cantidad": df.isnull().sum(),
    "Porcentaje (%)": round(df.isnull().mean()*100,2)
})

nulos = nulos[nulos["Cantidad"]>0]

nulos.sort_values("Cantidad", ascending=False)

,Cantidad,Porcentaje (%)
last_login_date,320,3.92
favorite_genre,240,2.94
monthly_watch_time_mins,193,2.37


En esta tabla se observan únicamente las variables que presentan valores faltantes. Esta información permitirá decidir posteriormente si corresponde eliminar registros, imputar valores o mantener la información sin modificaciones.

In [10]:
# Registros duplicados

duplicados = df.duplicated().sum()

print(f"Cantidad de registros duplicados: {duplicados}")

Cantidad de registros duplicados: 126


Los registros duplicados pueden afectar la calidad del análisis al representar varias veces la misma observación. En esta etapa únicamente se identifica su cantidad; cualquier decisión sobre su tratamiento será tomada durante la fase de limpieza.

In [11]:
# Variables categóricas

categoricas = df.select_dtypes(include="object").columns

tabla_categoricas = pd.DataFrame({

    "Variable": categoricas,

    "Cantidad de categorías": [df[col].nunique() for col in categoricas]

})

tabla_categoricas

,Variable,Cantidad de categorías
0,subscription_plan,15
1,country,26
2,favorite_genre,28
3,last_login_date,3062


In [12]:
# Primeras categorías encontradas

for variable in categoricas:

    print(f"\n{variable}")

    print(df[variable].unique()[:15])


subscription_plan
['Estándar' 'Básico' 'Premium' 'Std' 'estandar' 'basico' 'básico'
 'Premium ' 'premium' 'Premiun' 'BASICO' 'STANDARD' 'Basic' 'Estándar '
 'PREMIUM']

country
['Brasil' 'Colombia' 'Uruguay' 'Perú' 'Chile' 'Argentina' 'México'
 'Brazil' 'brasil' 'méxico' 'chile' 'uruguay' 'MEX' 'ARG' 'colombia']

favorite_genre
['Crime' 'Thriller' 'Drama' 'Acción' 'Romance' 'Comedia' 'Documental'
 'ACCIÓN' None 'CRIME' 'Comedia ' 'comedy' 'DRAMA' 'THRILLER'
 'Documentary']

last_login_date
['2025-03-04' '2019-04-02' '2018-04-13' '2021-01-31' '2020-09-30'
 '2020-07-03' '2019-07-26' '2019-02-24' '2025-08-03' '2024-02-12'
 '2018-12-23' '2024-08-15' '2022-09-25' '2018-05-11' '2018-05-03']


La inspección de las categorías permite detectar preliminarmente diferencias en la escritura de algunos valores. Se observan posibles inconsistencias relacionadas con mayúsculas, minúsculas, espacios adicionales, abreviaturas e incluso diferentes idiomas para representar una misma categoría. Estas situaciones serán analizadas durante la etapa de limpieza.

In [13]:
# Resumen general del dataset

resumen = pd.DataFrame({

    "Tipo": df.dtypes,

    "Valores nulos": df.isnull().sum(),

    "Valores únicos": df.nunique()

})

resumen

,Tipo,Valores nulos,Valores únicos
user_id,int64,0,8000
age,int64,0,69
subscription_plan,object,0,15
monthly_watch_time_mins,float64,193,5788
country,object,0,26
favorite_genre,object,240,28
last_login_date,object,320,3062
customer_support_tickets,int64,0,9


El resumen anterior reúne la información principal de cada variable, incluyendo su tipo de dato, la cantidad de valores faltantes y el número de categorías o valores distintos. Esta visión general será utilizada como referencia durante el proceso de limpieza y preparación de los datos.

## Observaciones iniciales

A partir de la inspección realizada se identificaron diversos aspectos que deberán analizarse con mayor detalle en la etapa de calidad y limpieza de datos.

Entre ellos se destacan:

- Presencia de valores faltantes en algunas variables.
- Posibles registros duplicados.
- Variables categóricas con diferentes formas de escritura para una misma categoría.
- Variables que podrían contener valores inconsistentes o atípicos.
- Necesidad de revisar el formato de algunas fechas.

Estas observaciones constituyen la evidencia inicial que permitirá justificar las decisiones de preparación del dataset en la siguiente etapa del proyecto.

# Preguntas de análisis

A partir de la inspección inicial surgieron las siguientes preguntas que orientarán el análisis exploratorio del proyecto:

1. ¿Cuál es el plan de suscripción más utilizado por los usuarios?

2. ¿Cómo se distribuye el tiempo mensual de visualización?

3. ¿Existen diferencias en el tiempo de visualización según el plan contratado?

4. ¿Existe relación entre la edad de los usuarios y el tiempo mensual de visualización?

5. ¿Qué relación presentan las variables numéricas del conjunto de datos?

# Conclusión

En esta primera etapa se realizó una inspección general del conjunto de datos con el propósito de comprender su estructura y evaluar su calidad inicial.

La exploración permitió identificar posibles valores faltantes, registros duplicados, inconsistencias en variables categóricas y otros aspectos que requerirán un análisis más detallado.
